In [ ]:
!pip install pandas --user
!pip install yfinance --user
!pip install numpy --user
!pip install matplotlib --user
!pip install pandas_datareader --users
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import datetime as date
import xgboost as xgb

In [ ]:
start = '2015-01-01'
today = '2024-05-01'
ticker= "AAPL"
data = yf.download(ticker, start, today)

In [ ]:
data


In [ ]:
data['Close'].plot()

In [ ]:
train_data=data.iloc[:int(.99*len(data)),:]
test_data=data.iloc[:int(.99*len(data)):,:]

In [ ]:
features =['Open','Volume']
target='Close'

In [ ]:
model = xgb.XGBRegressor()
model.fit(train_data[features], train_data[target])

In [ ]:
predictions=model.predict(test_data[features])
print('predictions')
print(predictions)

In [ ]:
print('Actual Values :')
print(test_data[target])

In [ ]:
accuracy= model.score(test_data[features], test_data[target])
print('Accuracy:')
print(accuracy)

In [ ]:
plt.plot(data['Close'],label='Close Price')
plt.plot(test_data[target].index, predictions, label='Predictions')
plt.legend()
plt.show()

In [ ]:
data = pd.DataFrame(data)


In [ ]:
data.reset_index(inplace=True)

In [ ]:
data

In [ ]:
data.drop(columns = ['Date','Open','High','Low','Volume'], inplace=True)

In [ ]:
data

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(data)

plt.show()

In [ ]:
ma100=data.Close.rolling(100).mean()
ma100

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(data.Close,'g')
plt.plot(ma100,'r')
plt.show()

In [ ]:
ma200=data.Close.rolling(200).mean()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(data.Close,'g')
plt.plot(ma200,'r')
plt.show()

In [ ]:
train_data = data[:-100]
test_data = data[-100:]

In [ ]:
train_data.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))


In [ ]:
train_data_scale = scaler.fit_transform(train_data)

In [ ]:
base_days = 100

In [ ]:
x = []
y = []
for i in range(base_days, train_data_scale.shape[0]):
    x.append(train_data_scale[i-base_days:i])
    y.append(train_data_scale[i,0])
   

In [ ]:
x[0]

In [ ]:
x, y = np.array(x), np.array(y)

In [ ]:

x = np.reshape(x, (x.shape[0],x.shape[1],1))


In [ ]:
!pip install tensorflow
!pip install keras

In [ ]:
from keras.layers import LSTM, Dense, Dropout
from keras.models import Sequential
   

In [ ]:
model = Sequential()
model.add(LSTM(50, activation = 'relu', return_sequences=True, input_shape=(x.shape[1],1)))
model.add(Dropout(0.2))

model.add(LSTM(units=60,activation='relu',return_sequences=True,
               ))
model.add(Dropout(0.3))

model.add(LSTM(units=80,activation='relu',return_sequences=True,))
model.add(Dropout(0.4))

model.add(LSTM(units=120,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(units=1))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer= 'adam',loss= 'mean_squared_error')

In [ ]:

model.fit(x,y,epochs=50,batch_size=32,verbose=1)

In [ ]:
test_data = pd.concat((train_data.tail(100), test_data), ignore_index=True)

In [ ]:
test_data

In [ ]:
test_data_scale = scaler.transform(test_data)

In [ ]:
x = []
y = []

for i in range(base_days, test_data_scale.shape[0]):
    x.append(test_data_scale[i-base_days:i])
    y.append(test_data_scale[i,0])

In [ ]:
x, y = np.array(x), np.array(y)

In [ ]:
x = np.reshape(x, (x.shape[0],x.shape[1],1))

In [ ]:
pred = model.predict(x)

In [ ]:
pred = scaler.inverse_transform(pred)

In [ ]:
pred

In [ ]:
m = y
z= []
future_days = 4
for i in range(base_days, len(m)+future_days):
    m = m.reshape(-1,1)
    inter = [m[-base_days:,0]]
    inter = np.array(inter)
    inter = np.reshape(inter, (inter.shape[0], inter.shape[1],1))
    pred = model.predict(inter)
    m = np.append(m ,pred)
    z = np.append(z, pred)

In [ ]:
z = np.array(z)
z = scaler.inverse_transform(z.reshape(-1,1))

In [ ]:
z

In [ ]:
model.save('stocPrice.keras')